# Week 18: Retrieval-Augmented Generation (RAG) - Part 2

## Measuring and Improving the Week 17 Pipeline

## Learning Objectives

By the end of this session, you will be able to:
1. **Compare chunking strategies** (fixed, recursive, hierarchical) and measure their impact on retrieval quality
2. **Add a Bedrock reranker** (Cohere Rerank 3.5) on top of a Strands retriever and quantify the precision lift
3. **Evaluate RAG outputs with RAGAS v0.4** (faithfulness, answer_relevancy, context_precision) using Bedrock Claude Haiku as the judge LLM - no external API keys
4. **Run an A/B comparison** of two RAG configurations end-to-end, pick a winner with data, and roll it back into the Week 17 supervisor

## Prerequisites

- Completed Week 17 (agentic RAG, Strands `policy_retriever_agent`, `mem0_memory`, Bedrock KB via `strands_tools.retrieve`)
- Comfortable with `strands_tools.retrieve`, Bedrock Converse API, SageMaker session + `get_execution_role()`
- Watched pre-class videos on chunking, reranking, RAGAS
- Have `STRANDS_KNOWLEDGE_BASE_ID` set (from Week 17 - same class KB)

## Session Format (~2 hours)

| Section | Duration | Type |
|---------|----------|------|
| Section 0: Setup & Week 17 Recap | 10 min | Code |
| Section 1: Chunking Strategies | 25 min | Demo + Lab 1 |
| Section 2: Reranking with Cohere Rerank 3.5 | 20 min | Demo + Lab 2 |
| Section 3: RAG Evaluation with RAGAS | 25 min | Demo + Lab 3 |
| Section 4: A/B Pipeline Optimization (MAIN OUTCOME) | 25 min | Demo + Main Lab 4 |
| Wrap-up & Homework | 5 min | Markdown |

## The Story So Far

In Week 17 you built a multi-retriever fraud supervisor. It WORKS, but how do you know it works WELL? Your manager asks: "Is our RAG pipeline reliable enough to replace the human-reviewed policy lookups?" You cannot answer with a demo. You need measurement.

This week you pull the three levers that move RAG quality in production:

```mermaid
graph LR
    subgraph "Week 17: WORKS"
        W17[policy_retriever_agent<br/>default chunking<br/>no reranking<br/>no evaluation]
    end

    subgraph "Week 18: WORKS WELL"
        C[Chunking<br/>fixed vs recursive<br/>vs hierarchical]
        R[Reranking<br/>Cohere Rerank 3.5]
        E[Evaluation<br/>RAGAS faithfulness<br/>answer_relevancy<br/>context_precision]
        AB[A/B Comparison<br/>pick a winner<br/>with numbers]
    end

    W17 --> C
    W17 --> R
    C --> AB
    R --> AB
    E --> AB
    AB --> OUT[Optimized supervisor<br/>defended with metrics]

    style OUT fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
```

The take-home is not a new agent. It is a TUNED version of Week 17's supervisor, plus the measurement discipline to defend it in a review.

## This Week vs Next Week

| | **Week 18 (Today)** | **Week 19 (Next Week)** |
|---|---|---|
| **Focus** | Measure and improve ONE RAG pipeline | Version and track MANY experiments |
| **You build** | A/B comparison DataFrame, pick winner | DVC-versioned data, MLflow-tracked runs |
| **Key concepts** | Chunking, reranking, RAGAS metrics | Data versioning, experiment tracking |
| **Why it matters** | You can defend your config with numbers | You can reproduce last month's config |

## GPU Setup

No GPU needed. All work is API-based through Amazon Bedrock. A CPU SageMaker Studio notebook is sufficient.

# Section 0: Environment Setup & Week 17 Recap

We reuse the Week 17 environment (Strands, Bedrock, SageMaker role) and add three new libraries:

- `langchain` + `langchain-aws` - for local chunking experiments and as the Bedrock bridge RAGAS needs
- `langchain-community` - for `BedrockEmbeddings`
- `ragas==0.4.3` - the evaluation framework, with Bedrock Claude Haiku as the judge

Nothing new for reranking - the Bedrock Rerank API is already reachable through `boto3.client('bedrock-agent-runtime').rerank(...)`.

AWS credentials come from your SageMaker execution role - same pattern as Week 17. No `getpass`, no API keys to paste.

The setup cells below verify every piece (model access, KB id, rerank model) and fail loud if anything is missing.

In [ ]:
# Install required libraries. Versions are lower-bound only so pip resolves
# quickly from cache. sagemaker is pinned to v2.x because v3 removed
# get_execution_role() from the top-level namespace.

%pip install -q \
    "sagemaker>=2.200,<3" \
    "strands-agents>=1.37" \
    "strands-agents-tools[mem0-memory]>=0.2.10" \
    "boto3>=1.35" \
    "langchain>=0.3" \
    "langchain-aws>=0.2" \
    "langchain-community>=0.3" \
    "ragas>=0.4" \
    "datasets>=2.18" \
    "faiss-cpu>=1.9" \
    "rank_bm25>=0.2.2"

print("\nPackages installed. If this was your first install, RESTART THE KERNEL before running the next cell.")

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
# IMPORTANT: strands_tools.retrieve reads env vars AT IMPORT TIME. We set the
# required ones (KNOWLEDGE_BASE_ID, MIN_SCORE, RETRIEVE_ENABLE_METADATA_DEFAULT)
# BEFORE the strands import so the tool works without a kernel restart.

import os
import json
import time
import boto3
import sagemaker
import pandas as pd
from sagemaker import get_execution_role
from importlib.metadata import version as pkg_version


# =============================================================================
# SET strands_tools.retrieve ENVIRONMENT VARIABLES (before import)
# =============================================================================
os.environ["KNOWLEDGE_BASE_ID"] = os.environ.get("KNOWLEDGE_BASE_ID") \
    or os.environ.get("STRANDS_KNOWLEDGE_BASE_ID") \
    or "FARSQGTONR"  # same class KB as Week 17
os.environ["STRANDS_KNOWLEDGE_BASE_ID"] = os.environ["KNOWLEDGE_BASE_ID"]
os.environ["MIN_SCORE"] = os.environ.get("MIN_SCORE", "0.2")
os.environ["RETRIEVE_ENABLE_METADATA_DEFAULT"] = "true"


# =============================================================================
# STRANDS + LANGCHAIN IMPORTS
# =============================================================================
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import retrieve

# Chunking + local indexing (NEW in Week 18)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings, ChatBedrockConverse
from langchain_core.documents import Document

# RAGAS (NEW in Week 18)
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Version check
for pkg in ["strands-agents", "boto3", "langchain", "langchain-aws", "ragas", "faiss-cpu", "numpy"]:
    try:
        print(f"  {pkg:22s} {pkg_version(pkg)}")
    except Exception:
        print(f"  {pkg:22s} (not installed)")


# =============================================================================
# SAGEMAKER SESSION + EXECUTION ROLE (same pattern as Week 15/16/17)
# =============================================================================
sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name

os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

print(f"\nSageMaker execution role: {role.split('/')[-1]}")
print(f"AWS Region:               {AWS_REGION}")


# =============================================================================
# MODEL CONFIGURATION (same as Week 15/16/17)
# =============================================================================
MODEL_ID       = "us.anthropic.claude-3-haiku-20240307-v1:0"
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"

llm = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION)

langchain_llm      = ChatBedrockConverse(model=MODEL_ID, region_name=AWS_REGION)
bedrock_embeddings = BedrockEmbeddings(model_id=EMBED_MODEL_ID, region_name=AWS_REGION)

print(f"\nLLM:        {MODEL_ID}")
print(f"Embeddings: {EMBED_MODEL_ID}")
print(f"Reranker:   Claude Haiku listwise (same model, no marketplace required)")


# =============================================================================
# PRE-FLIGHT PROBES
# =============================================================================
# Probe 1: LLM access
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
try:
    probe = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 10, "temperature": 0},
    )
    print(f"\nLLM probe OK:    {probe['output']['message']['content'][0]['text']!r}")
except Exception as e:
    print(f"\nLLM probe FAILED: {e}")
    print(f"Ask your instructor to enable Bedrock access for {MODEL_ID}.")
    raise


# Probe 2: Week 17 shared Knowledge Base id
STRANDS_KNOWLEDGE_BASE_ID = os.environ["KNOWLEDGE_BASE_ID"]

bedrock_agent = boto3.client("bedrock-agent", region_name=AWS_REGION)
try:
    kb_info = bedrock_agent.get_knowledge_base(knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID)
    print(f"KB probe OK:     {STRANDS_KNOWLEDGE_BASE_ID} ({kb_info['knowledgeBase']['name']})")
except Exception as e:
    print(f"KB probe FAILED: {e}")
    print("Ask your instructor for the correct Knowledge Base id.")
    raise

# bedrock-agent-runtime client (used for direct retrieve + rerank calls)
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)

print("\nEnvironment ready.")

In [ ]:
# =============================================================================
# WEEK 17 RECAP - BASELINE POLICYRETRIEVER (no rerank, default chunking)
# =============================================================================
import logging
logging.getLogger("mem0").setLevel(logging.CRITICAL)

baseline_policy_retriever = Agent(
    model=llm,
    tools=[retrieve],
    system_prompt=(
        "You are a fraud policy specialist. Always call retrieve before "
        "answering. Answer only from retrieved content. If the KB does not "
        "cover the question, say so and stop."
    ),
    callback_handler=None,
)


# =============================================================================
# MEM0 MEMORY CONFIGURATION (for Section 5 multi-agent outcome)
# =============================================================================
# mem0_memory stores per-investigator case history. FAISS backend writes to
# /tmp - ephemeral on SageMaker Studio, which is fine for class sessions.
# user_id scopes memory to one investigator.
os.environ["MEM0_LLM_PROVIDER"]   = "aws_bedrock"
os.environ["MEM0_LLM_MODEL"]      = MODEL_ID
os.environ["MEM0_EMBED_PROVIDER"] = "aws_bedrock"
os.environ["MEM0_EMBED_MODEL"]    = EMBED_MODEL_ID

from strands_tools import mem0_memory

# =============================================================================
# LOCAL FRAUD CORPUS - same content as the Week 17 KB, as in-memory text
# so we can freely experiment with chunking. In production you would read
# this from S3; here we inline it for classroom use.
# =============================================================================
FRAUD_POLICY_CORPUS = [
    ("ctr_rules.md",
     "Currency Transaction Report (CTR) rules under 31 CFR 1010.311. Banks "
     "must file a CTR for each transaction in currency of more than $10,000. "
     "Multiple transactions are aggregated when known to be conducted by or "
     "on behalf of the same person and result in cash in or cash out totaling "
     "more than $10,000 in any one business day. Structuring - breaking a "
     "transaction into smaller amounts to evade the CTR threshold - is itself "
     "a federal violation under 31 USC 5324."),
    ("ofac_screening.md",
     "OFAC screening requirements. All wire transfers must be screened against "
     "the OFAC Specially Designated Nationals (SDN) list before execution. "
     "International wire transfers involving countries on the OFAC sanctions "
     "list (including but not limited to Iran, North Korea, Syria, and Cuba) "
     "require additional review and may be blocked outright. False positives "
     "must be cleared within 24 hours."),
    ("wire_record_keeping.md",
     "Wire transfer recordkeeping under 31 CFR 1010.410. For any international "
     "wire transfer of $3,000 or more, the bank must retain the originator's "
     "name, address, account number, amount, execution date, payment "
     "instructions, beneficiary bank, and beneficiary name. Records must be "
     "retained for five years."),
    ("structuring_red_flags.md",
     "Structuring red flags. Multiple cash deposits of amounts just under "
     "$10,000 across consecutive days at the same or related accounts are a "
     "classic structuring pattern. Velocity anomalies - for example three or "
     "more transactions at unrelated merchants within 30 minutes - indicate "
     "potential card testing or account takeover."),
    ("account_takeover_patterns.md",
     "Account takeover (ATO) indicators. A password change followed within "
     "minutes by a wire transfer to a newly added payee from an unfamiliar IP "
     "is a high-confidence ATO signal. Transactions that originate from "
     "geographies inconsistent with the customer's historical footprint, "
     "especially from countries the customer has never transacted with, "
     "require hold and verification."),
    ("unusual_hours_rule.md",
     "Unusual hours rule. Transactions initiated between 1:00 AM and 5:00 AM "
     "local time that fall outside the customer's typical active window "
     "require enhanced monitoring. Two or more such transactions within a "
     "single session should trigger a SAR review."),
    ("new_payee_hold.md",
     "New payee large transfer hold. Wire transfers exceeding $5,000 to payees "
     "that were added to the account within the preceding 72 hours require "
     "two-factor customer verification and a 24-hour hold regardless of the "
     "customer's risk score."),
    ("high_risk_merchant_categories.md",
     "High-risk merchant category codes (MCCs). Cryptocurrency exchanges, "
     "offshore gambling platforms, and money transfer services are classified "
     "as high-risk MCCs. Transactions at these merchants for amounts over "
     "$1,000 require enhanced due diligence."),
]

print("Baseline policy retriever ready.")
print(f"Local fraud corpus loaded: {len(FRAUD_POLICY_CORPUS)} policy documents.")
print("mem0_memory tool configured for Section 5.")

# Section 1: Chunking Strategies

## Why Chunking Is a Lever

A chunk is the unit a retriever fetches. If the chunk is too small, the LLM loses context. If the chunk is too large, the retriever loses precision and costs rise. The 2026 published benchmark (Vecta, 50 academic papers) puts recursive character splitting at 512 tokens with 50-100 overlap at 69% end-to-end accuracy, outperforming more exotic methods. But NVIDIA's FinanceBench benchmark found 1024-token chunks outperformed 512 on financial documents - because tables and numerical context need to stay together.

Fraud policies look a lot like financial prose. Short general answer: start with recursive-512-80, but measure.

```mermaid
graph TB
    subgraph "Fixed"
        F1[300 tokens<br/>10% overlap] --> FX[Many small chunks<br/>loses cross-section context]
    end

    subgraph "Recursive"
        R1[512 tokens<br/>80 token overlap<br/>split on paragraph -> line -> space] --> RX[Benchmark default<br/>respects structure]
    end

    subgraph "Hierarchical"
        H1[Parent 1500 + Child 300<br/>retrieve child, return parent] --> HX[Production AWS default<br/>best retrieval + best context]
    end

    style RX fill:#e8f5e9,stroke:#4caf50
    style HX fill:#e8f5e9,stroke:#4caf50
```

## Bedrock Native Chunking Options

Bedrock KBs accept a `ChunkingConfiguration` at ingestion time:

- `NONE` - treat each file as one chunk
- `FIXED_SIZE` - pick tokens + overlap percentage
- `HIERARCHICAL` - parent + child token budgets
- `SEMANTIC` - split on natural-language similarity boundaries (extra cost - uses an FM)

For this section we experiment LOCALLY with langchain splitters because we cannot re-ingest the shared class KB mid-lesson. In production you would pass the same parameters to Bedrock's `ChunkingConfiguration` at KB creation.

In [ ]:
# =============================================================================
# DEMO: Split the Fraud Corpus Three Ways and Index Locally
# =============================================================================
# We build three local FAISS indexes over the SAME text so we can isolate
# the effect of chunking alone. The embedding model is Titan V2 in all
# three indexes. The only thing that changes is HOW we chop the text.

# Materialize the corpus as langchain Documents
documents = [Document(page_content=text, metadata={"source": name})
             for name, text in FRAUD_POLICY_CORPUS]

# ---- Config A: small fixed chunks ----
splitter_a = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks_a   = splitter_a.split_documents(documents)

# ---- Config B: benchmark default (recursive 512/80) ----
splitter_b = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=80)
chunks_b   = splitter_b.split_documents(documents)

# ---- Config C: fraud-domain tuned (recursive 1024/150) ----
splitter_c = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=150)
chunks_c   = splitter_c.split_documents(documents)

print(f"Config A (300/30):   {len(chunks_a)} chunks")
print(f"Config B (512/80):   {len(chunks_b)} chunks")
print(f"Config C (1024/150): {len(chunks_c)} chunks")

# Index each with Titan V2 embeddings into a local FAISS store
index_a = FAISS.from_documents(chunks_a, bedrock_embeddings)
index_b = FAISS.from_documents(chunks_b, bedrock_embeddings)
index_c = FAISS.from_documents(chunks_c, bedrock_embeddings)

print("\nThree FAISS indexes built with Titan V2 embeddings.")

In [ ]:
# =============================================================================
# DEMO: Query Each Chunk Config with Two Test Queries
# =============================================================================
# Test queries chosen to surface a difference:
#   Q1 is a direct lookup ("CTR threshold")       - small chunks should win
#   Q2 needs cross-section context ("password change then wire transfer")
#                                                 - larger chunks should win
#                                                   because they keep the
#                                                   sequence together

queries = [
    ("Q1 direct lookup",
     "What is the CTR reporting threshold?"),
    ("Q2 multi-context",
     "What indicates account takeover when a password change is followed by a wire transfer?"),
]

for label, q in queries:
    print(f"\n=== {label}: {q!r} ===")
    for name, idx in [("A (300/30)",   index_a),
                      ("B (512/80)",   index_b),
                      ("C (1024/150)", index_c)]:
        hits = idx.similarity_search_with_score(q, k=1)
        doc, score = hits[0]
        preview = doc.page_content[:120].replace("\n", " ")
        print(f"  {name:15s} score={score:6.3f}  source={doc.metadata['source']}")
        print(f"                  text: {preview}...")

print("\nObservation: config choice depends on query shape.")
print("The lab below makes that observation measurable.")

## Lab 1: Chunking Trade-off Table (15 min)

### Your Task

Build a pandas DataFrame comparing how the THREE chunk configs from the demo perform across FOUR fraud test queries. This is the first time you produce metrics in this week's notebook.

### Steps

1. Define a list of 4 test queries covering different shapes (direct lookup, cross-section, multi-document, and an adversarial one the KB likely cannot answer).
2. For each (query, config) pair, retrieve the top-1 hit. Record:
   - config name
   - query label
   - retrieved source filename
   - similarity score
3. Assemble a pandas DataFrame with columns: `query`, `config`, `source`, `score`.
4. Print the DataFrame sorted by query so you can eyeball which config wins per query shape.

### Expected Output

A 12-row DataFrame (4 queries x 3 configs) with readable scores. You should see config C win on queries needing more context, config A win on direct lookups. Not every query has a single correct config.

### Stretch (for fast finishers)

Add a 4th config using `MarkdownHeaderTextSplitter` on headings. The fraud policy docs in the demo corpus use file-based separation that could be exploited by a header-aware splitter. Measure whether it wins on any query and explain why.

### Homework Extension

Replicate ONE of your local configs in Bedrock itself by creating a small second KB with `ChunkingConfiguration=HIERARCHICAL` (parent 1500, child 300, overlap 60). Ingest the same 8 policy docs. Compare retrieval quality against your local results. Why might Bedrock's hierarchical retrieval score DIFFERENTLY than local recursive splitting even though the words are identical?

In [ ]:
# =============================================================================
# SOLUTION: Lab 1 - Chunking Trade-off Table
# =============================================================================
# Four queries covering different shapes: direct lookup (small chunks win),
# cross-section (large chunks win), multi-policy, and adversarial (not in KB).
# The adversarial query tests whether the model hallucinates when the KB lacks
# the answer - an important compliance check.

lab1_queries = [
    ("direct_lookup",  "What is the CTR reporting threshold?"),
    ("cross_section",  "What indicates account takeover after a password change?"),
    ("multi_policy",   "What are the wire transfer recordkeeping requirements?"),
    ("adversarial",    "What is the $50,000 suspicious activity threshold?"),
]

rows = []
for label, q in lab1_queries:
    for cfg_name, idx in [("A (300/30)", index_a),
                          ("B (512/80)", index_b),
                          ("C (1024/150)", index_c)]:
        hits = idx.similarity_search_with_score(q, k=1)
        doc, score = hits[0]
        rows.append({
            "query":  label,
            "config": cfg_name,
            "source": doc.metadata["source"],
            "score":  round(score, 4),
        })

lab1_df = pd.DataFrame(rows).sort_values("query")
print(lab1_df.to_string(index=False))

# Expected observation:
# - direct_lookup: config A (300/30) wins - small chunks isolate the $10,000 sentence
# - cross_section: config C (1024/150) wins - keeps the password-change + wire sequence together
# - adversarial: all configs return something (no "not found") - this is why faithfulness
#   evaluation is critical; the model may hallucinate an answer from a nearby chunk

> **Think About It #1**: Your Lab 1 table shows config C wins on multi-context queries and config A wins on direct lookups. In production you cannot ship three retrievers - you ship one. Do you pick the config that wins on the most QUERIES, or the config whose LOSSES are least damaging for compliance? (Hint: in a regulated domain, a missed policy reference is worse than an extra chunk of irrelevant text.) How would you decide, and what would you write in the PR description to justify it?

# Section 2: Reranking - The Highest-ROI Addition

## Bi-encoder vs Cross-encoder

The retriever you have been using is a BI-ENCODER: it embeds the query once, embeds each chunk once, computes cosine similarity. Fast, but imperfect at deciding which of the top-10 is MOST relevant to THIS specific query.

A RERANKER is a CROSS-ENCODER: it looks at (query, chunk) as a pair and scores relevance directly. Much more accurate on the top few, at the cost of roughly 50-100 ms per candidate.

The production pattern:

```mermaid
graph LR
    Q[Query] --> RET[Bi-encoder retrieve<br/>top 20]
    RET --> RR[LLM listwise reranker<br/>Claude Haiku]
    RR --> TOP3[top 3 by relevance]
    TOP3 --> LLM[Generator LLM]

    style RR fill:#e8f5e9,stroke:#4caf50,stroke-width:2px
```

Published 2026 guidance: reranking is the single highest-ROI addition to a basic RAG pipeline, typically 10-30% precision gain for 50-100 ms latency.

## Listwise LLM Reranking with Claude Haiku

We implement reranking as a LISTWISE LLM call: all candidate chunks are sent to
Claude Haiku in a single `converse` call with an explicit ranking instruction.
Haiku returns a JSON array of `{index, relevance_score}` pairs sorted by relevance.

Advantages of this approach:
- No separate reranker model subscription required
- One API call for the entire candidate list (not one per chunk)
- Teaches HOW reranking works - the ranking logic is visible and editable
- Identical output shape to a dedicated reranker, so you can swap in Cohere
  Rerank 3.5 later by changing one function without touching the rest of the pipeline

Note: a dedicated cross-encoder (like Cohere Rerank 3.5 on Bedrock) will generally
be more accurate than an LLM prompted for ranking, especially on domain-specific text.
The listwise LLM approach is a solid production pattern and an excellent learning
vehicle - it makes the cross-encoder intuition concrete before you swap in a
dedicated model.

In [ ]:
# =============================================================================
# DEMO: LLM-Based Listwise Reranking (Claude Haiku)
# =============================================================================
# We use Claude Haiku as a cross-encoder alternative. All candidate chunks
# are sent in ONE converse call with a ranking prompt. Haiku returns a JSON
# array of {index, relevance_score} pairs - same output shape as a
# dedicated reranker model. No marketplace subscription required.
#
# Latency: 200-400 ms per rerank call (one API call, not one per chunk).
# This is ALSO a teaching moment: you can implement reranking with any LLM
# that follows instructions, not just a dedicated reranker endpoint.

import json as _json

def bedrock_rerank(query_text: str, text_sources: list,
                   num_results: int = 3) -> list:
    """Rerank text_sources by relevance to query_text using Claude Haiku listwise ranking.

    All candidates are sent in one converse call. Haiku returns a ranked
    JSON array. Same output shape as a dedicated reranker endpoint:
      [{index, relevance_score, text}, ...] sorted by relevance_score desc.

    Args:
        query_text:   The query to rank against.
        text_sources: List of plain-text chunk strings to rank.
        num_results:  How many top results to return.

    Returns:
        List of dicts with keys: index (0-indexed into text_sources),
        relevance_score (0.0-1.0), text.
    """
    if not text_sources:
        return []

    # Cap at 10 candidates to stay within prompt budget
    sources_to_rank = text_sources[:10]
    sources_text = "\n".join(
        f"{i+1}. {src[:300]}..." if len(src) > 300 else f"{i+1}. {src}"
        for i, src in enumerate(sources_to_rank)
    )

    prompt = (
        "You are a relevance ranking expert. Given a query and a list of "
        "documents, rank them by relevance to the query.\n\n"
        f"Query: {query_text}\n\n"
        f"Documents:\n{sources_text}\n\n"
        "Return a JSON array with ALL documents ranked from most to least "
        "relevant. Include the original document number (1-indexed) and a "
        "relevance score from 0.0 to 1.0.\n\n"
        "Example format:\n"
        '[{"index": 2, "relevance_score": 0.95}, '
        '{"index": 1, "relevance_score": 0.72}, '
        '{"index": 3, "relevance_score": 0.40}]\n\n'
        "Return ONLY a valid JSON array, no other text."
    )

    try:
        response = bedrock_runtime.converse(
            modelId=MODEL_ID,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            inferenceConfig={"maxTokens": 500, "temperature": 0.0},
        )
        response_text = response["output"]["message"]["content"][0]["text"]
        ranked = _json.loads(response_text)

        result = []
        for item in ranked:
            if isinstance(item, dict) and "index" in item:
                idx = item["index"] - 1  # convert 1-indexed -> 0-indexed
                if 0 <= idx < len(sources_to_rank):
                    result.append({
                        "index":           idx,
                        "relevance_score": float(item.get("relevance_score", 0.0)),
                        "text":            sources_to_rank[idx],
                    })

        # Sort desc and truncate to num_results
        return sorted(result,
                      key=lambda x: x["relevance_score"],
                      reverse=True)[:num_results]

    except (_json.JSONDecodeError, KeyError, TypeError):
        # Fallback: return sources in original order with decaying scores
        print("bedrock_rerank: JSON parse failed, returning original order.")
        return [
            {"index": i, "relevance_score": 1.0 - (i * 0.1), "text": s}
            for i, s in enumerate(sources_to_rank[:num_results])
        ]


# ----- Demo: try it on 5 candidates from index_b -----
q = "What is the CTR reporting threshold for cash transactions?"
candidates = [doc.page_content for doc, _ in
              index_b.similarity_search_with_score(q, k=5)]

ranked = bedrock_rerank(q, candidates, num_results=3)
for r in ranked:
    print(f"rank_score={r['relevance_score']:.3f}  idx_in_candidates={r['index']}")
    print(f"  text: {r['text'][:160].replace(chr(10), ' ')}...\n")

In [ ]:
# =============================================================================
# DEMO: Measure the Rerank Lift on a Deliberately Ambiguous Query
# =============================================================================
# We pick a query where the bi-encoder's top hit is NOT the best chunk,
# then show the Haiku listwise reranker promoting the better chunk to rank 1.

q = "When must a bank report a large cash deposit to the government?"

# Bi-encoder baseline top-5
baseline = index_b.similarity_search_with_score(q, k=5)
print("BASELINE bi-encoder top-5 (index_b, 512/80 chunks):")
for i, (doc, score) in enumerate(baseline, 1):
    print(f"  #{i}  score={score:.3f}  source={doc.metadata['source']}")

# Rerank those 5 using the Haiku listwise reranker
candidates = [doc.page_content for doc, _ in baseline]
ranked = bedrock_rerank(q, candidates, num_results=5)
print("\nAFTER Haiku listwise reranking (same 5 candidates, reordered):")
sources = [baseline[r["index"]][0].metadata["source"] for r in ranked]
for i, r in enumerate(ranked, 1):
    print(f"  #{i}  rerank_score={r['relevance_score']:.3f}  source={sources[i-1]}")

print("\nObservation: the chunks do not change, but their order does.")
print("The reranker moved the most policy-relevant chunk to position 1.")

## Lab 2: Rerank the Week 17 PolicyRetriever (15 min)

### Your Task

Wrap the Week 17 retrieval step with Cohere Rerank 3.5. Show before/after output for TWO queries where you expect reranking to help.

### Steps

1. Write a function `reranked_retrieve(query, k_retrieve=10, k_final=3)` that:
   - Calls the Week 17 agent's retrieve tool with `k_retrieve` results
   - Extracts the chunks from the tool output
   - Calls `bedrock_rerank(...)` to reorder them
   - Returns the top `k_final` chunks (with their rerank scores and source URIs)
2. Pick TWO queries that you expect reranking to help on (multi-hop or slightly ambiguous). Run each query with and without reranking. Print the top result for each.
3. Observe which query benefits MORE from reranking. In a code comment below the output, write one sentence explaining why.

### Expected Output

Two before/after comparisons, each showing the top-1 chunk with and without reranking, and a one-sentence observation.

### Stretch (for fast finishers)

Time both versions with `time.perf_counter`. Report the latency overhead of reranking per query. In which production scenario is the extra latency unacceptable (think about user-facing chat vs. batch compliance review)?

### Homework Extension

Replace the `retrieve` tool in Week 17's `policy_retriever_agent` with a custom `@tool` that calls `reranked_retrieve` under the hood. Verify the agent still answers the same questions correctly. Does the agent SAY it used reranking? If not, how would you surface that to an auditor?

In [ ]:
# =============================================================================
# SOLUTION: Lab 2 - Bolt Reranking onto the Week 17 Retriever
# =============================================================================
# Key insight: we MUST call bedrock_agent_runtime.retrieve() directly, not
# strands_tools.retrieve. The Strands tool wraps its output in a ToolResult
# envelope and returns a pre-formatted string. bedrock_rerank() needs a plain
# list of text strings to score. The raw Bedrock API gives us retrievalResults
# which we can unpack into candidates + source location dicts.

def reranked_retrieve(query, k_retrieve=10, k_final=3):
    """Retrieve from the Week 17 KB then rerank with Cohere 3.5."""
    # Step 1: raw retrieval - get k_retrieve candidates from Bedrock KB
    resp = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": k_retrieve}},
    )
    items = resp.get("retrievalResults", [])

    # Step 2: extract plain text and source location from each result
    candidates = [i.get("content", {}).get("text", "") for i in items]
    sources    = [i.get("location", {}) for i in items]

    # Step 3: rerank - cross-encoder scores the (query, chunk) pairs
    ranked = bedrock_rerank(query, candidates, num_results=k_final)

    # Attach source location to each ranked result so callers can cite it
    for r in ranked:
        r["source"] = sources[r["index"]]
    return ranked


# Before/after comparison on 2 queries that benefit from reranking
lab2_queries = [
    "When must a bank file a CTR for aggregated transactions?",
    "What new payee wire transfer rules apply for recently added payees?",
]

for q in lab2_queries:
    print(f"\nQuery: {q!r}")

    # Baseline: top-1 from bi-encoder only
    baseline_hits = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": q},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
    ).get("retrievalResults", [])
    before = baseline_hits[0].get("content", {}).get("text", "")[:120] if baseline_hits else "no results"
    print(f"  BEFORE rerank: {before}")

    # After: top-1 after Cohere rerank
    ranked = reranked_retrieve(q, k_retrieve=10, k_final=3)
    after  = ranked[0]["text"][:120] if ranked else "no results"
    score  = ranked[0]["relevance_score"] if ranked else 0
    print(f"  AFTER  rerank: {after}")
    print(f"          score: {score:.3f}")

# Observation: the aggregated-CTR query benefits more because the bi-encoder
# retrieves the structuring doc (semantically close) but the cross-encoder
# correctly promotes the CTR aggregation rule to rank 1.

> **Think About It #2**: Reranking adds 50-100 ms per query. In a fraud investigation workflow where the agent makes 3-5 retrieve calls per case, that is 150-500 ms of added latency per case. For an investigator reviewing 200 cases/day, is that worth the precision gain? What would change your answer: case volume, missed-policy cost, SLA contracts?

# Section 3: RAG Evaluation with RAGAS v0.4

## From "It Works on My Demo" to Numbers

Up to now our assessment of RAG quality has been "this looks right." That does not survive a compliance audit. RAGAS is an evaluation framework that scores RAG outputs with LLM-as-judge metrics.

## The Three Metrics We Will Use

| Metric | What it asks | Range | When it fails |
|--------|--------------|-------|---------------|
| `faithfulness` | Is the answer supported by the retrieved context? | 0-1 | Model hallucinated beyond what was retrieved |
| `answer_relevancy` | Does the answer address the user's question? | 0-1 | Model answered a different question |
| `context_precision` | Are retrieved chunks actually relevant? | 0-1 | Retriever returned off-topic chunks |

A fourth metric `context_recall` needs GROUND-TRUTH references for each question and is deferred to homework.

## LLM-as-Judge Architecture

```mermaid
graph LR
    Q[Question] --> R[Retriever]
    R --> C[Retrieved Context]
    C --> G[Generator LLM<br/>Claude Haiku]
    G --> A[Answer]

    Q --> J[RAGAS Judge LLM<br/>also Claude Haiku]
    C --> J
    A --> J
    J --> S[Scores:<br/>faithfulness<br/>answer_relevancy<br/>context_precision]

    style J fill:#fff3e0,stroke:#ff9800,stroke-width:2px
```

We use Claude Haiku for both the generator AND the judge. This is a common cost-efficient pattern and is documented in the AWS RAGAS + Bedrock sample. More robust setups use a stronger judge model, which we discuss in the wrap-up.

## Bedrock + RAGAS Setup

RAGAS needs a judge LLM and an embedder. We wrap LangChain's Bedrock clients so RAGAS can call them:

```python
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

evaluator_llm        = LangchainLLMWrapper(langchain_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(bedrock_embeddings)
```

Note on RAGAS v0.4: `LangchainLLMWrapper` is scheduled for deprecation in a future version in favor of `llm_factory()`. We use the wrapper here because it is the stable documented path in the AWS sample notebook.

In [ ]:
# =============================================================================
# DEMO: Score One Retriever Answer with RAGAS
# =============================================================================
# RAGAS needs an EvaluationDataset. Each sample is a SingleTurnSample with
# user_input, response, and retrieved_contexts. We produce those three
# fields ourselves by calling bedrock-agent-runtime.retrieve() (for raw
# chunks) + Claude generator.

# Wrap the LangChain Bedrock clients for RAGAS
evaluator_llm        = LangchainLLMWrapper(langchain_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(bedrock_embeddings)

q = "What is the CTR reporting threshold?"

# Call Bedrock retrieve directly (Strands tool returns a pre-formatted string;
# we need the raw retrievalResults list for RAGAS context).
resp = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID,
    retrievalQuery={"text": q},
    retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
)
retrieved_contexts = [it.get("content", {}).get("text", "")
                      for it in resp.get("retrievalResults", [])]

# Generate an answer (the same way the agent would, but direct)
gen_prompt = (
    "Answer using only the context.\n\n"
    f"Context:\n{chr(10).join(retrieved_contexts)}\n\nQuestion: {q}\nAnswer:"
)
answer = langchain_llm.invoke(gen_prompt).content

# Assemble a 1-sample EvaluationDataset
sample = SingleTurnSample(
    user_input         = q,
    response           = answer,
    retrieved_contexts = retrieved_contexts,
)
dataset = EvaluationDataset(samples=[sample])

# Evaluate
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)
print("RAGAS result for single sample:")
print(result)

## Lab 3: RAGAS Score Sheet for 5 Fraud Questions (15 min)

### Your Task

Build an evaluation dataset of 5 fraud questions, run the baseline PolicyRetriever + Claude generator against each, and score with RAGAS on `faithfulness` and `answer_relevancy` (skip `context_precision` for speed). Produce a pandas DataFrame showing per-question scores.

### Steps

1. Author 5 fraud questions that cover different topics (CTR, OFAC, ATO, structuring, high-risk MCC). Keep them narrow.
2. Write a helper `build_sample(question)` that retrieves + generates + returns a `SingleTurnSample`.
3. Build an `EvaluationDataset` from the 5 samples.
4. Evaluate on `[faithfulness, answer_relevancy]`.
5. Convert the result to a pandas DataFrame and sort by faithfulness.

### Expected Output

A 5-row DataFrame with columns: question, faithfulness, answer_relevancy. Scores below 0.7 on either metric are worth investigating.

### Stretch (for fast finishers)

Write 3 ADVERSARIAL questions - questions that sound reasonable but the KB does not cover (e.g., "What is the $50,000 threshold rule?" which is a made-up rule). Score them. What does the agent do when asked something the KB cannot answer? Does `answer_relevancy` catch it?

### Homework Extension

Write a function `gate(df, min_faith=0.8, min_rel=0.7)` that returns `True` only if the MINIMUM score across all rows passes BOTH thresholds. This is a CI-style deployment gate. Wrap it around the week's evaluation so that a config must pass before you promote it. Which metric threshold do you trust more for a compliance use case, and why?

In [ ]:
# =============================================================================
# SOLUTION: Lab 3 - RAGAS Score Sheet for 5 Fraud Questions
# =============================================================================
# build_sample calls bedrock_agent_runtime.retrieve() directly because
# RAGAS needs a list of plain strings for retrieved_contexts, not the
# formatted ToolResult string that strands_tools.retrieve returns.
# Using the raw Bedrock API gives us retrievalResults which we unpack
# into the three RAGAS fields: user_input, response, retrieved_contexts.

def build_sample(question: str) -> SingleTurnSample:
    """Retrieve + generate an answer + wrap into a SingleTurnSample for RAGAS."""
    # Step 1: raw retrieval (not strands_tools.retrieve - need plain text list)
    resp = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": question},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
    )
    contexts = [i.get("content", {}).get("text", "")
                for i in resp.get("retrievalResults", [])]

    # Step 2: generate an answer grounded in the retrieved context
    prompt = (
        "Answer using only the context.\n\n"
        f"Context:\n{chr(10).join(contexts)}\n\n"
        f"Question: {question}\nAnswer:"
    )
    answer = langchain_llm.invoke(prompt).content

    # Step 3: package as a RAGAS SingleTurnSample
    return SingleTurnSample(
        user_input=question,
        response=answer,
        retrieved_contexts=contexts,
    )


lab3_questions = [
    "What is the CTR reporting threshold for cash transactions?",
    "What OFAC screening is required before a wire transfer?",
    "What account takeover indicators follow a password change?",
    "When is structuring a federal violation?",
    "What wire transfer recordkeeping rules apply to $3,000 transactions?",
]

# Build evaluation dataset (5 API calls to both retrieve and generate)
lab3_dataset = EvaluationDataset(samples=[build_sample(q) for q in lab3_questions])

# Score: faithfulness = no hallucination; answer_relevancy = answered the question
lab3_result = evaluate(
    dataset=lab3_dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

lab3_df = lab3_result.to_pandas()[["user_input", "faithfulness", "answer_relevancy"]]
lab3_df = lab3_df.rename(columns={"user_input": "question"})
lab3_df = lab3_df.sort_values("faithfulness")
print(lab3_df.to_string(index=False))

# Any row below 0.7 on faithfulness is a hallucination candidate worth investigating.
# answer_relevancy below 0.7 means the model answered a different question.
low_faith = lab3_df[lab3_df["faithfulness"] < 0.7]
if not low_faith.empty:
    print(f"\n{len(low_faith)} question(s) have faithfulness < 0.7 - worth investigating.")
else:
    print("\nAll questions passed faithfulness >= 0.7.")

> **Think About It #3**: Claude Haiku is both the generator AND the judge in your Lab 3. That is cheap and common, but it is also a form of self-evaluation. Name two failure modes this could hide. When would you pay for Claude Sonnet (or another model) as the judge, and how would you budget for it?

# Section 4: A/B Pipeline Optimization (MAIN OUTCOME)

## Combine the Three Levers

You have three levers: chunking, reranking, evaluation. This section combines them into a single A/B comparison that tells you WHICH configuration wins and by HOW MUCH.

## The Comparison Matrix

```mermaid
graph TB
    subgraph "Config 1: Baseline"
        B1[Bedrock KB<br/>default retrieval<br/>no rerank]
    end

    subgraph "Config 2: Tuned"
        B2[Bedrock KB<br/>default retrieval<br/>+ Cohere Rerank 3.5]
    end

    B1 --> EVAL[RAGAS<br/>faithfulness<br/>answer_relevancy]
    B2 --> EVAL
    EVAL --> DF[pandas DataFrame<br/>config x metric<br/>pick winner]

    style DF fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
```

This is the ML engineer's core workflow: propose configurations, measure them against a fixed eval dataset, pick the winner, justify the choice with numbers.

## What We Deliberately Keep Simple This Week

- Only 2 configs (baseline vs reranked). Adding more is the stretch.
- Only 3 questions. More is the stretch.
- Only 2 metrics. Adding `context_precision` is the stretch.
- No experiment tracking tool (MLflow/DVC) - Week 19's job.
- No A/B test with live traffic - Week 20's job (Langfuse observability).

This is the starting discipline, not the end state.

In [ ]:
# =============================================================================
# DEMO: Evaluate Baseline vs Reranked Side by Side
# =============================================================================
# Reuse the `build_sample` function from Lab 3 as the baseline generator.
# Create a `build_sample_reranked` variant that uses `reranked_retrieve`.
# Score each on 3 questions, produce a single comparison DataFrame.

def build_sample_reranked(question):
    ranked = reranked_retrieve(question, k_retrieve=10, k_final=3)
    contexts = [r["text"] for r in ranked]
    prompt = (
        "Answer using only the context.\n\n"
        f"Context:\n{chr(10).join(contexts)}\n\n"
        f"Question: {question}\nAnswer:"
    )
    answer = langchain_llm.invoke(prompt).content
    return SingleTurnSample(
        user_input=question,
        response=answer,
        retrieved_contexts=contexts,
    )


demo_questions = [
    "What is the CTR reporting threshold?",
    "When must a bank file a CTR on aggregated transactions?",
    "What evidence indicates account takeover?",
]


def score_config(config_name, sample_builder, questions):
    """Score one RAG config across a list of questions. Returns a DataFrame."""
    ds = EvaluationDataset(samples=[sample_builder(q) for q in questions])
    r = evaluate(
        dataset=ds,
        metrics=[faithfulness, answer_relevancy],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
    )
    df = r.to_pandas()
    df["config"] = config_name
    return df[["config", "user_input", "faithfulness", "answer_relevancy"]]


base_scores   = score_config("baseline", build_sample,           demo_questions)
rerank_scores = score_config("+ rerank", build_sample_reranked,  demo_questions)
compare = pd.concat([base_scores, rerank_scores], ignore_index=True)
print(compare.to_string(index=False))

print("\nMean scores per config:")
print(compare.groupby("config")[["faithfulness", "answer_relevancy"]]
               .mean().round(3))

## Main Lab 4: A/B Winner + Writeup (15 min)

### Your Task

This is the Week 18 main outcome. Pick TWO configs, evaluate them against a shared eval set, decide a winner, and write one paragraph explaining the choice to a reviewer who will not run your notebook.

### Steps

1. Define `lab4_eval_questions` - at least 3 fraud questions you care about.
2. Define two configurations to compare:
   - Config A: baseline (reuse `build_sample` from Lab 3)
   - Config B: a configuration of your choice (reranked is the obvious one, but you could also chunk locally at 1024/150 and retrieve from `index_c`, etc.)
3. Score both configs on the eval set with `[faithfulness, answer_relevancy]`.
4. Produce a `lab4_comparison_df` with columns: `config`, `question`, `faithfulness`, `answer_relevancy`.
5. Print the winner - the config with the higher MEAN across both metrics.
6. In a MARKDOWN CELL below your code cell, write 3-4 sentences explaining which config won, by how much, and why you would (or would not) ship it.

### Expected Output

- A 6-row DataFrame (2 configs x 3 questions)
- A printed "winner: ..." line
- A short written justification you would paste into a PR description

### Stretch (for fast finishers)

Add a third config (e.g. local chunking `index_c` with Cohere rerank on top). Add `context_precision` as a third metric. Add a `latency_ms` column by timing each sample's end-to-end production. Does the "best" config change when you factor latency in?

### Homework Extension (CRITICAL - closes the Week 17 loop)

Take the WINNER of your Lab 4 comparison and roll it into Week 17's `lab3_supervisor`. Create a new `@tool` called `optimized_policy_retrieve` that implements the winning config, and register it as the PolicyRetriever's underlying retrieval function. Re-run Lab 4 on the UPDATED supervisor to confirm the lift survives the agentic loop. Write one paragraph on whether the lift is bigger, smaller, or unchanged once the supervisor is in the loop, and WHY.

In [ ]:
# =============================================================================
# SOLUTION: Main Lab 4 - A/B Winner + Writeup
# =============================================================================
# Config A: baseline retrieval (no reranking)
# Config B: Cohere Rerank 3.5 on top of the same KB retrieval
#
# We reuse build_sample (baseline) and build_sample_reranked (defined in the
# Section 4 demo) for clean isolation. The only difference between configs
# is whether Cohere reranks the candidates before the LLM sees them.

lab4_eval_questions = [
    "What is the CTR reporting threshold for cash transactions?",
    "When must a bank screen a wire transfer against OFAC?",
    "What new payee wire transfer rules apply within 72 hours of adding a payee?",
]


def score_config_lab4(config_name, sample_builder, questions):
    """Score one RAG config. Returns a clean DataFrame with a config column."""
    ds = EvaluationDataset(samples=[sample_builder(q) for q in questions])
    r = evaluate(
        dataset=ds,
        metrics=[faithfulness, answer_relevancy],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
    )
    df = r.to_pandas()
    df["config"] = config_name
    return df[["config", "user_input", "faithfulness", "answer_relevancy"]].rename(
        columns={"user_input": "question"}
    )


baseline_scores = score_config_lab4("A: baseline",   build_sample,           lab4_eval_questions)
rerank_scores   = score_config_lab4("B: + rerank",   build_sample_reranked,  lab4_eval_questions)

lab4_comparison_df = pd.concat([baseline_scores, rerank_scores], ignore_index=True)
print(lab4_comparison_df.to_string(index=False))

# Pick winner by mean across both metrics
means = lab4_comparison_df.groupby("config")[["faithfulness", "answer_relevancy"]].mean()
means["combined"] = means.mean(axis=1)
winner = means["combined"].idxmax()
print(f"\nMean scores per config:")
print(means.round(3))
print(f"\nWinner: {winner}")

> **Think About It #4**: You have numbers now. The baseline vs reranked comparison is probably a small lift (a few tenths on each metric). In a real code review, a reviewer will ask: "Why should we pay the 50-100 ms per query and the per-query Cohere fee for that lift?" What is the minimum lift you would demand before shipping a RAG change to production at a financial institution, and what non-numeric evidence (compliance, auditability, risk) would you bring to the review?

# Section 5: Full Multi-Agent Supervisor (MAIN OUTCOME)

## Closing the Week 17-18 Arc

You have measured three RAG levers. Now combine them into a production-grade supervisor
that you can carry into Weeks 19-20 for MLOps versioning.

The Week 18 outcome is a supervisor with TWO tools:

1. `reranked_policy_retrieve` - the optimized retrieval tool from Lab 2 (Bedrock KB +
   Haiku listwise reranker). This replaces the raw `retrieve` tool from Week 17.
2. `mem0_memory` - per-investigator case history (same as Week 17 Lab 2's case_history_agent).

```mermaid
graph TB
    INV[Investigator Query] --> SUP[week18_supervisor<br/>Claude Haiku 3]

    subgraph "Tool 1: Optimized KB"
        SUP --> RPR[reranked_policy_retrieve<br/>Bedrock KB + Haiku listwise reranker]
        RPR --> POLICY[Fraud Policy<br/>+ Compliance Rules]
    end

    subgraph "Tool 2: Case Memory"
        SUP --> MEM[mem0_memory<br/>FAISS backend]
        MEM --> CASES[Prior Case<br/>Findings]
    end

    RPR --> ANS[Supervisor Answer<br/>with policy citations]
    MEM --> ANS

    style ANS fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
```

The supervisor is also the backend for `fraud_agent_ui.py` - a ~50-line Gradio app
you can run from the SageMaker terminal to chat with the agent in a browser.

In [ ]:
# =============================================================================
# DEMO: Week 18 Full Supervisor (reranked KB + mem0 memory)
# =============================================================================
# This is the Week 18 main outcome. The pattern is identical to Week 17's
# lab3_supervisor, but the policy retrieval tool now uses reranked_retrieve
# so every KB lookup is Haiku-listwise-reranked.
#
# We wrap reranked_retrieve in a @tool so Strands can call it.

from strands import tool as strands_tool

@strands_tool
def reranked_policy_retrieve(query: str) -> str:
    """Retrieve fraud policies from the Bedrock KB and rerank with Claude Haiku.

    Args:
        query: The investigation question or policy lookup.

    Returns:
        The top 3 policy chunks ranked by relevance, as a formatted string.
    """
    ranked = reranked_retrieve(query, k_retrieve=10, k_final=3)
    lines = []
    for i, r in enumerate(ranked, 1):
        src_uri = r.get("source", {}).get("s3Location", {}).get("uri", "unknown")
        lines.append(f"[{i}] score={r['relevance_score']:.3f} | {src_uri}")
        lines.append(r["text"])
        lines.append("")
    return "\n".join(lines)


week18_supervisor = Agent(
    model=llm,
    tools=[reranked_policy_retrieve, mem0_memory],
    system_prompt=(
        "You are a senior fraud investigator. For every question: "
        "1) Search your memory for prior findings on this case or customer. "
        "2) Retrieve the relevant fraud policies using reranked_policy_retrieve. "
        "3) Synthesize a clear compliance recommendation with policy citations. "
        "Always save your key findings to memory after answering."
    ),
    callback_handler=None,
)

print("week18_supervisor ready.")
print("Tools: reranked_policy_retrieve, mem0_memory")

# Demo: two-turn session to show memory works across turns
print("\n--- Turn 1 ---")
r1 = week18_supervisor(
    "Customer made 3 cash deposits of $9,500 each over 3 days. "
    "Is this structuring? User: investigator_demo",
    user_id="investigator_demo",
)
print(str(r1)[:500])

print("\n--- Turn 2 (memory should recall the structuring finding) ---")
r2 = week18_supervisor(
    "What did I find about this customer earlier? User: investigator_demo",
    user_id="investigator_demo",
)
print(str(r2)[:400])

## Running the Fraud Agent UI

`week18_supervisor` above is the backend for a standalone Gradio chat app.

Run it from the SageMaker terminal:

```bash
python fraud_agent_ui.py
```

Or from a notebook cell:

```python
!python fraud_agent_ui.py &
```

Gradio will print a URL like `http://0.0.0.0:7860`. In SageMaker Studio, access
it through the Studio proxy URL that Gradio prints at startup.

`fraud_agent_ui.py` is in the same folder as this notebook. It is self-contained -
no imports from this notebook. You can carry it directly into Weeks 19-20 for
DVC versioning and CI/CD integration.

---

**The arc is complete:**
- Week 17: built the pipeline (agentic RAG + case memory)
- Week 18: measured and improved it (chunking + reranking + RAGAS + full supervisor)
- Week 19: version and track it (DVC + MLflow)
- Week 20: observe it in production (Langfuse)

The `fraud_agent_ui.py` is what you are versioning next week.

## Optional Notebooks (in this folder)

- `week_18_optional_hybrid_search.ipynb` - BM25 + FAISS + Reciprocal Rank Fusion.
  No new installs needed (`rank_bm25` is already installed).
- `week_18_optional_deep_eval.ipynb` - DeepEval 6-metric evaluation suite with
  a custom Claude Haiku 3 judge (bypasses the native BedrockModel class which
  targets Sonnet-class model IDs not available in the class account).

# Summary: What We Learned Today

## Key Takeaways

### Chunking
- Recursive 512/80 is the benchmark default; 1024/150 wins on cross-section fraud/finance queries. Measure, do not assume.
- Bedrock Knowledge Bases accept FIXED / HIERARCHICAL / SEMANTIC / NONE as `ChunkingConfiguration`. Hierarchical + hybrid search + reranking is the AWS-recommended production default.

### Reranking
- We implemented listwise LLM reranking using Claude Haiku via a single `converse` call.
- Same output shape as a dedicated reranker: `{index, relevance_score, text}` sorted by relevance.
- One API call for all candidates (not one per chunk) - 200-400 ms total latency.
- The listwise LLM pattern makes the cross-encoder intuition concrete. In production you can
  swap in Cohere Rerank 3.5 on Bedrock (us-east-1) by replacing the function body - the
  interface stays identical.

### RAGAS v0.4
- Faithfulness, answer_relevancy, context_precision as the main three.
- `LangchainLLMWrapper(ChatBedrockConverse(...))` as the judge - no OpenAI.
- `context_recall` needs ground-truth references - homework.

### A/B Optimization
- Two configs, shared eval set, pick a winner by mean metric score.
- The discipline is the deliverable: you can defend your config with data.

### Full Multi-Agent Supervisor (Section 5)
- `week18_supervisor` holds `reranked_policy_retrieve` + `mem0_memory` as tools.
- Same pattern as Week 17's lab3_supervisor, now with measured retrieval quality under it.
- Run `fraud_agent_ui.py` from the SageMaker terminal to chat with this supervisor
  in a browser UI. Carry it into Weeks 19-20 for DVC versioning.

## Looking Ahead

- **Week 19 (MLOps Part 1)** introduces DVC for data versioning and MLflow for experiment tracking. Every comparison DataFrame you built today would live in MLflow next week, and every eval dataset would be DVC-versioned.
- **Week 20 (MLOps Part 2)** adds online observability with Langfuse and LiteLLM around the exact same supervisor. Today's RAGAS scores are OFFLINE; Week 20's are ONLINE - on real production traffic.
- **Weeks 21-22 (Airflow)** schedule periodic re-evaluation runs so your RAGAS numbers stay fresh as the KB grows.
- **Week 23 (Ethics)** connects today's `faithfulness` metric to GDPR and EU AI Act audit requirements - citations from retrieve + RAGAS faithfulness scores are exactly what an auditor will ask for.
- **Week 24 (Capstone)** - an evaluated, reranked, retrieval-augmented multi-agent system is one of the three canonical capstone paths.

## Homework

### Homework 1: Bedrock HIERARCHICAL Chunking (Lab 1 extension)
Create a second Bedrock KB with `ChunkingConfiguration=HIERARCHICAL` (parent 1500, child 300, overlap 60). Ingest the same 8 policy docs. Compare retrieval quality vs your local Lab 1 results. Why might Bedrock's hierarchical retrieval score DIFFERENTLY than local recursive splitting?

### Homework 2: Latency-Aware Rerank (Lab 2 extension)
Time your `reranked_retrieve` with `time.perf_counter`. Report latency overhead per query. For a fraud supervisor that makes 3-5 retrieve calls per case and serves 200 cases/day, what is the daily wall-clock cost of reranking?

### Homework 3: CI Gate + context_recall (Lab 3 extension)
Implement `gate(df, min_faith=0.8, min_rel=0.7) -> bool`. Also add `context_recall` as a fourth metric - this requires authoring ground-truth references per question. Decide: which of the four metrics do you put on the CI gate, and which do you track but not block on?

### Homework 4: Roll Winner into Week 17 Supervisor (Lab 4 extension - CRITICAL)
Take the winner of your Lab 4 comparison. Create a new `@tool` called `optimized_policy_retrieve` and wire it into Week 17's `policy_retriever_agent` in place of the raw `retrieve`. Re-run Week 17's Lab 3 multi-retriever supervisor end-to-end on the full fraud workflow. Does the supervisor's final decision quality change? Write one paragraph on whether the offline lift (what we measured this week) survives the agent loop (what Week 17 built). This is the closing loop of the RAG arc.

## Great Work Today

**The journey so far:**
- Weeks 11-14: Make LLMs respond (prompt, fine-tune, classify)
- Weeks 15-16: Make LLMs ACT (tools, memory, supervisor multi-agent)
- Week 17: Make LLMs LOOK THINGS UP (agentic RAG with Bedrock KB + FAISS)
- **Week 18: Measure and improve what they look up (chunking + reranking + RAGAS)**

**Next up**: Week 19 - MLOps for the RAG pipeline you just tuned.